In [1]:
import sys, torch
print(sys.executable)
assert torch.cuda.is_available(), "GPU 미사용!"
print(torch.cuda.get_device_name(0))

C:\projects\PycharmProjects\PythonProject\RunModel\.venv\Scripts\python.exe
NVIDIA GeForce RTX 2060


In [2]:
import os
print(os.getcwd())

C:\projects\PycharmProjects\PythonProject\RunModel


In [3]:
from datasets import load_dataset
from dotenv import load_dotenv
import os


load_dotenv()
dataset = load_dataset("Cheukting/math-meta-reasoning-cleaned", token=os.getenv("HUGGINGFACEHUB_API_TOKEN"))
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'token_count'],
        num_rows: 987485
    })
})

In [4]:
from transformers import GPT2Tokenizer


tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_function(examples):
   return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [5]:
tokenized_datasets_split = tokenized_datasets["train"].shard(num_shards=100, index=0).train_test_split(test_size=0.2, shuffle=True)
tokenized_datasets_split

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'token_count', 'input_ids', 'attention_mask'],
        num_rows: 7900
    })
    test: Dataset({
        features: ['id', 'text', 'token_count', 'input_ids', 'attention_mask'],
        num_rows: 1975
    })
})

In [6]:
# import torch, time
# from transformers import GPT2LMHeadModel
#
# m = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").cuda()
# opt = torch.optim.AdamW(m.parameters(), lr=1e-5)
# scaler = torch.amp.GradScaler()
# x = torch.randint(0, 50257, (4, 512), device="cuda")
#
# for i in range(20):
#     if i == 5:                      # 워밍업 5스텝은 버림
#         torch.cuda.synchronize(); t = time.time()
#     with torch.autocast("cuda", dtype=torch.float16):
#         loss = m(x, labels=x).loss
#     scaler.scale(loss).backward()
#     scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
#
# torch.cuda.synchronize()
# print(f"{(time.time()-t)/15*1000:.0f} ms/step (배치 4)")
# print(f"실제 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")
# print(f"예약(캐시 포함): {torch.cuda.memory_reserved()/1e9:.2f} GB")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


216 ms/step (배치 4)
실제 사용: 1.53 GB
예약(캐시 포함): 5.20 GB


## GPT2 모델 세부 조정

이러한 인수 중 대부분은 모델 세부 조정에서 표준적으로 사용하는 값입니다. 다만 사용하는 컴퓨터 환경에 따라 몇 가지를 조정할 필요가 있습니다.

1. 배치 크기 – 최적의 배치 크기를 찾는 것이 중요합니다. 배치 크기가 클수록 트레이닝 속도가 빨라집니다. 그러나 CPU나 GPU 메모리에는 한계가 있기 때문에 일정 수준 이상은 불가능할 수 있습니다.
2. 에포크 – 에포크 수가 많아질수록 트레이닝 시간이 길어집니다. 필요한 에포크 수는 직접 결정할 수 있습니다.
3. 저장 단계 – 저장 단계를 설정하면 체크포인트가 디스크에 저장되는 주기가 결정됩니다. 트레이닝이 느리고, 중간에 예기치 않게 멈출 수 있다면 더 자주 저장하도록 설정하는 것이 좋습니다(더 낮은 값으로 설정).

In [ ]:
## 트레이닝 인수 설정
from transformers import TrainingArguments

# 버전 1
training_args = TrainingArguments(
    output_dir="./results",
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    fp16=True,
    logging_steps=20,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
)

In [ ]:
from transformers import GPT2LMHeadModel, Trainer, DataCollatorForLanguageModeling


model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_datasets_split['train'],
   eval_dataset=tokenized_datasets_split['test'],
   data_collator=data_collator,
)


trainer.train(resume_from_checkpoint=False)

트레이닝이 끝나면 모델을 평가하고 저장한다.

저장한 모델은 파이프라인에서 사용 가능하다.

In [ ]:
trainer.evaluate(tokenized_datasets_split['test'])

In [ ]:
trainer.save_model("./trained_model")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
model.push_to_hub("gpt2-math-reasoning", private=True)
tokenizer.push_to_hub("gpt2-math-reasoning", private=True)

# model.push_to_hub("sookyeong/gpt2-finetuned")
# tokenizer.push_to_hub("sookyeong/gpt2-finetuned")

## 두 번째 모델 학습

In [6]:
# import torch
# print(torch.__version__, "| CUDA:", torch.version.cuda)
# print(torch.cuda.get_device_name(0))
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# print("시퀀스 길이:", len(tokenized_datasets_split['train'][0]['input_ids']))

2.9.0+cu129 | CUDA: 12.9
NVIDIA GeForce RTX 2060
VRAM: 6.4 GB
시퀀스 길이: 512


In [7]:
# del m, opt, scaler, x, loss
# import gc; gc.collect()
# torch.cuda.empty_cache()
# print(f"정리 후: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [6]:
from transformers import TrainingArguments

training_args2 = TrainingArguments(
    output_dir='./results3',
    num_train_epochs=2,                  # 3 → 1 (학습용이면 충분)
    per_device_train_batch_size=4,       # 4 → 8 (GPU 활용률↑)
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,       # 유효 배치는 16으로 유지
    warmup_steps=100,
    weight_decay=0.01,
    save_steps=300,
    save_total_limit=2,                  # ← 디스크 보호, 아래 설명
    logging_steps=100,
    fp16=True,                           # ← 가장 큰 효과
    # dataloader_pin_memory=False,       # ← CUDA에선 삭제 (기본값 True가 맞음)
)

In [7]:
from transformers import GPT2LMHeadModel, Trainer, DataCollatorForLanguageModeling

model2 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


trainer2 = Trainer(
   model=model2,
   args=training_args2,
   train_dataset=tokenized_datasets_split['train'],
   eval_dataset=tokenized_datasets_split['test'],
   data_collator=data_collator,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [8]:
trainer2.train(resume_from_checkpoint=False)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.109097
200,1.658208
300,1.577882
400,1.485022
500,1.434837
600,1.420081
700,1.373883
800,1.380864
900,1.340878
1000,1.331434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1976, training_loss=1.3893369064639938, metrics={'train_runtime': 885.9653, 'train_samples_per_second': 17.834, 'train_steps_per_second': 2.23, 'total_flos': 4128414105600000.0, 'train_loss': 1.3893369064639938, 'epoch': 2.0})

In [9]:
trainer2.evaluate(tokenized_datasets_split['test'])

Training Loss,Validation Loss,Step
1.249986,1.185492,1976


{'eval_loss': 1.1854918003082275}

In [10]:
trainer2.save_model("./trained_model3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
model2.push_to_hub("gpt2-math-reasoning", private=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Hyosong/gpt2-math-reasoning/commit/5ef0ed7dee3b346ada643adabf1fabfe10bc25fd', commit_message='Upload model', commit_description='', oid='5ef0ed7dee3b346ada643adabf1fabfe10bc25fd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Hyosong/gpt2-math-reasoning', endpoint='https://huggingface.co', repo_type='model', repo_id='Hyosong/gpt2-math-reasoning'), pr_revision=None, pr_num=None)

In [17]:
# tokenizer.push_to_hub("gpt2-math-reasoning", private=True)

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Hyosong/gpt2-math-reasoning/commit/72aacaed9ba9fc017dfa05d9f9a2942ac0a1adfd', commit_message='Upload tokenizer', commit_description='', oid='72aacaed9ba9fc017dfa05d9f9a2942ac0a1adfd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Hyosong/gpt2-math-reasoning', endpoint='https://huggingface.co', repo_type='model', repo_id='Hyosong/gpt2-math-reasoning'), pr_revision=None, pr_num=None)